# 05c — Distancia a la línea de pobreza/indigencia, era histórica (Tornarolli 2018, CEDLAS DT 226)

Extiende `data/02b_distancia_linea_pobreza/hogar/` a la era histórica, usando la canasta
retropolada construida y validada en `05b_canastas_retropoladas.ipynb` y la misma
función de distancia que ya usa la era regular, agnóstica de era.

In [ ]:
import sys
import time
from pathlib import Path


def _raiz_repo(desde: Path | None = None) -> Path:
    actual = (desde or Path.cwd()).resolve()
    for candidato in (actual, *actual.parents):
        if (candidato / "pytest.ini").exists():
            return candidato
    raise FileNotFoundError(f"no se encontró la raíz del repo subiendo desde {actual}")


REPO = _raiz_repo()
sys.path.insert(0, str(REPO / "src"))

import pandas as pd

import pipeline_meta as pm
from armonizacion import trimestres_publicados
import insumos_lineas as il
import distancia_linea_pobreza
import canasta_retropolada as cr
import ponderadores

NUCLEO = REPO / "data" / "02_nucleo"
ARMONIZADO = REPO / "data" / "01_armonizado"
META = REPO / "data" / "meta"
CANASTAS = REPO / "data" / "canastas_indec"
INSUMOS = REPO / "data" / "insumos_lineas"
RETROPOLACION = CANASTAS / "retropolacion"
COMMIT_GIT = pm.obtener_commit_git(REPO)

TRIMESTRES_HISTORICOS = [t for t in trimestres_publicados() if t <= (2016, 1)]
print(f"{len(TRIMESTRES_HISTORICOS)} trimestres históricos publicados "
      f"({min(TRIMESTRES_HISTORICOS)}-{max(TRIMESTRES_HISTORICOS)})")

## Canasta retropolada (re-derivada en memoria)

No se lee el CSV que escribe `05b` en `data/meta/` (gitignored) -- se vuelve a derivar
acá, para que esta notebook sea regenerable por sí sola.

In [ ]:
ancla_dic_2006 = cr.cargar_ancla_dic_2006(INSUMOS / "dic_2006_gba_metodologia22.csv")
ancla_abr_2016 = cr.cargar_ancla_abr_2016(CANASTAS)

indec_largo = il.cargar_ipc_gba_historico(INSUMOS / "ipc_gba_historico_1943_2008_crudo.csv")
indec_nivel_general = indec_largo[indec_largo["serie"] == "ipc_gba_nivel_general"].set_index("mes")["valor"]
cifra_nivel_general = il.cargar_ipc_provincias_cifra(
    INSUMOS / "ipc_provincias_cifra_2007_2018_crudo.xlsx").set_index("mes")["valor"]
indice, variacion_empalme_ene_2007 = cr.encadenar_indice(indec_nivel_general, cifra_nivel_general)

cba_gba_mensual = cr.construir_canasta_gba_mensual(
    ancla_dic_2006["cba_adulto_equivalente"], ancla_abr_2016.set_index("REGION").loc[cr.REGION_GBA, "cba_adulto_equivalente"], indice, "cba_adulto_equivalente")
cbt_gba_mensual = cr.construir_canasta_gba_mensual(
    ancla_dic_2006["cbt_adulto_equivalente"], ancla_abr_2016.set_index("REGION").loc[cr.REGION_GBA, "cbt_adulto_equivalente"], indice, "cbt_adulto_equivalente")

cba_regiones_mensual = cr.extender_regiones(cba_gba_mensual, ancla_abr_2016, "cba_adulto_equivalente")
cbt_regiones_mensual = cr.extender_regiones(cbt_gba_mensual, ancla_abr_2016, "cbt_adulto_equivalente")
cba_regiones_trimestral = cr.agregar_trimestral_por_origen(cba_regiones_mensual, "cba_adulto_equivalente")
cbt_regiones_trimestral = cr.agregar_trimestral_por_origen(cbt_regiones_mensual, "cbt_adulto_equivalente")

canasta_trimestral_retropolada = cba_regiones_trimestral.merge(
    cbt_regiones_trimestral, on=["ANO4", "TRIMESTRE", "REGION", "origen"], suffixes=("", "_cbt"))
print(f"{len(canasta_trimestral_retropolada)} filas región×trimestre×origen")
canasta_trimestral_retropolada["origen"].value_counts()

## Control de cobertura de la canasta filtrada

Se valida que la serie de dos anclas, la recomendada por `05b`, cubra todo el período
histórico sin huecos. Si el supuesto no se sostiene, no se toca el filtro ni
`src/canasta_retropolada.py` sin avisar.

In [ ]:
canasta_historica = canasta_trimestral_retropolada[
    canasta_trimestral_retropolada["origen"] == cr.ORIGEN_DOS_ANCLAS].copy()

pares_cubiertos = canasta_historica[["ANO4", "TRIMESTRE"]].drop_duplicates().sort_values(["ANO4", "TRIMESTRE"])
primer_trimestre = tuple(pares_cubiertos.iloc[0])
ultimo_trimestre = tuple(pares_cubiertos.iloc[-1])
incluye_2003_2006 = bool((pares_cubiertos["ANO4"] < 2007).any())

esperados = [(a, t) for a in range(2003, 2017) for t in range(1, 5) if (2003, 3) <= (a, t) <= (2016, 1)]
cubiertos = set(map(tuple, pares_cubiertos.values.tolist()))
faltantes = [p for p in esperados if p not in cubiertos]

print(f"primer trimestre cubierto (origen=dos_anclas): {primer_trimestre}")
print(f"último trimestre cubierto (origen=dos_anclas): {ultimo_trimestre}")
print(f"incluye 2003-2006: {incluye_2003_2006}")
print(f"faltantes dentro de 2003T3-2016T1: {faltantes}")

assert incluye_2003_2006, "2003-2006 no está bajo origen=dos_anclas -- PARAR, revisar con el usuario antes de seguir"
assert not faltantes, f"huecos inesperados dentro de 2003T3-2016T1: {faltantes} -- PARAR, revisar con el usuario"

El primer trimestre cubierto por la canasta arranca antes de lo que necesita la EPH --
la canasta sobra por ese lado, no falta nada. El control pasa.

## Distancia por hogar -- piloto (sin escritura en disco)

Mismo patrón que `05`/`05b`: columnas sin `PONDIH` (que no existe en la era histórica)
y `REGION` traída desde `01_armonizado`.

In [ ]:
COLUMNAS_HOGAR_HIST = ["CODUSU", "NRO_HOGAR", "ANO4", "TRIMESTRE", "AGLOMERADO", "ITF", "ingreso_no_declarado"]
COLUMNAS_IND_HIST = ["CODUSU", "NRO_HOGAR", "COMPONENTE", "CH04", "CH06", "PONDERA"]
ERA = "historica"
ANIO_PILOTO, TRIMESTRE_PILOTO = 2006, 4


def construir_distancia_historica(anio: int, trimestre: int) -> pd.DataFrame:
    hogar = pm.cargar_hogar(anio, trimestre, COLUMNAS_HOGAR_HIST, NUCLEO)
    individual = pm.cargar_individual(anio, trimestre, COLUMNAS_IND_HIST, NUCLEO)
    hogar = pm.agregar_region(hogar, ARMONIZADO, anio, trimestre)
    canasta_t = canasta_historica[(canasta_historica["ANO4"] == anio) & (canasta_historica["TRIMESTRE"] == trimestre)]

    resultado = distancia_linea_pobreza.calcular_distancia_linea_pobreza(hogar, individual, canasta_t[["REGION", "cba_adulto_equivalente", "cbt_adulto_equivalente"]])

    peso_hogar = ponderadores.peso_hogar_historico(individual)
    n_miembros = individual.groupby(["CODUSU", "NRO_HOGAR"]).size().rename("n_miembros")
    idx = pd.MultiIndex.from_frame(resultado[["CODUSU", "NRO_HOGAR"]])
    resultado["peso_hogar"] = peso_hogar.reindex(idx).to_numpy()
    resultado["n_miembros"] = n_miembros.reindex(idx).to_numpy()
    resultado["ANO4"], resultado["TRIMESTRE"], resultado["era"] = anio, trimestre, ERA
    return resultado


distancia_piloto = construir_distancia_historica(ANIO_PILOTO, TRIMESTRE_PILOTO)
print(f"{len(distancia_piloto)} hogares, {ANIO_PILOTO}T{TRIMESTRE_PILOTO}")

### Tratamiento de los hogares con `ingreso_no_declarado`

`distancia_linea_pobreza.calcular_distancia_linea_pobreza` anula `ratio_linea_pobreza`/`ratio_linea_indigencia` (los deja `NA`) solo
cuando el flag es exactamente `True` -- la fila del hogar nunca se excluye del
resultado ni se imputa, queda con su marca intacta para que quien agregue una tasa
decida cómo tratarla. En la era histórica el flag es siempre `NA` (no hay `PONDIH` de
hogar antes de 2016), y un `NA` no enmascara nada. `itf_cero_declarado` tampoco se
excluye ni se imputa: son pobres/indigentes reales si el ratio cae por debajo de 1.

In [ ]:
n = len(distancia_piloto)
ingreso_no_declarado_true = (distancia_piloto["ingreso_no_declarado"] == True).fillna(False)  # noqa: E712

print(f"n hogares: {n}")
print(f"% REGION nula: {100 * distancia_piloto['REGION'].isna().mean():.2f}%")
print(f"% unidades_adulto_equivalente NA: {100 * distancia_piloto['unidades_adulto_equivalente'].isna().mean():.2f}%")
print(f"% ingreso_no_declarado == True: {100 * ingreso_no_declarado_true.mean():.2f}%")
print(f"% ingreso_no_declarado == NA: {100 * distancia_piloto['ingreso_no_declarado'].isna().mean():.2f}%")
print(f"% itf_cero_declarado: {100 * (distancia_piloto['itf_cero_declarado'] == True).mean():.2f}%")
print()
print(f"canasta usada por región, {ANIO_PILOTO}T{TRIMESTRE_PILOTO}:")
canasta_historica[
    (canasta_historica["ANO4"] == ANIO_PILOTO) & (canasta_historica["TRIMESTRE"] == TRIMESTRE_PILOTO)
][["REGION", "cba_adulto_equivalente", "cbt_adulto_equivalente"]].sort_values("REGION")

## Tasa de pobreza/indigencia propia del trimestre vs. Tornarolli (Anexo B)

Comparación informal, no la validación completa de más abajo (que es por semestre
completo y restringe a los mismos aglomerados que usa Tornarolli): CEDLAS reporta
semestres, no trimestres, así que un trimestre suelto se compara contra el semestre
que lo incluye a título de referencia.

In [ ]:
tasa_ponderada = distancia_linea_pobreza.tasa_ponderada


usable = distancia_piloto[~ingreso_no_declarado_true].copy()
usable["peso_personas"] = usable["peso_hogar"] * usable["n_miembros"]

pobreza_personas = tasa_ponderada(usable, "pobre", "peso_personas")
indigencia_personas = tasa_ponderada(usable, "indigente", "peso_personas")

cedlas = pd.read_csv(RETROPOLACION / "cedlas" / "series_comparables_anexo_b.csv")
fila_cedlas = cedlas[(cedlas["ANO4"] == ANIO_PILOTO) & (cedlas["SEMESTRE"] == 2)].iloc[0]

print(f"pobreza personas, {ANIO_PILOTO}T{TRIMESTRE_PILOTO} (propia): {pobreza_personas:.2f}%")
print(f"pobreza personas, {ANIO_PILOTO}-II (Tornarolli/CEDLAS, serie comparable): {fila_cedlas['pob_serie_comparable']:.2f}%")
print(f"indigencia personas, {ANIO_PILOTO}T{TRIMESTRE_PILOTO} (propia): {indigencia_personas:.2f}%")
print(f"indigencia personas, {ANIO_PILOTO}-II (Tornarolli/CEDLAS, serie comparable): {fila_cedlas['indig_serie_comparable']:.2f}%")

## Corrida completa -- todos los trimestres históricos, escritura en disco

Extiende el cálculo a todos los trimestres históricos disponibles y escribe cada
partición en `data/02b_distancia_linea_pobreza/hogar/` -- aditivo, nunca pisa las particiones de
la era regular. Columnas al disco: iguales a las de esa capa salvo `PONDIH` (no existe
en la era histórica), reemplazada por `peso_hogar` + `n_miembros`.

In [ ]:
DISTANCIA_LINEA_POBREZA = REPO / "data" / "02b_distancia_linea_pobreza" / "hogar"
DISTANCIA_LINEA_POBREZA.mkdir(parents=True, exist_ok=True)

COLUMNAS_DISCO = ["CODUSU", "NRO_HOGAR", "ANO4", "TRIMESTRE", "REGION", "AGLOMERADO", "era",
                  "ITF", "peso_hogar", "n_miembros", "unidades_adulto_equivalente", "cba_adulto_equivalente", "cbt_adulto_equivalente",
                  "linea_indigencia", "linea_pobreza", "ratio_linea_pobreza", "brecha_relativa_linea_pobreza",
                  "log_ratio_linea_pobreza", "ratio_linea_indigencia", "brecha_relativa_linea_indigencia", "log_ratio_linea_indigencia",
                  "pobre", "indigente", "ingreso_no_declarado", "itf_cero_declarado"]

filas_cobertura = []
n_filas_totales = 0
t0 = time.time()
for anio, trimestre in TRIMESTRES_HISTORICOS:
    assert (anio, trimestre) < (2016, 2), f"{(anio, trimestre)} no es histórico -- no se escribe"
    resultado = construir_distancia_historica(anio, trimestre)
    ingreso_no_declarado_true = (resultado["ingreso_no_declarado"] == True).fillna(False)  # noqa: E712
    filas_cobertura.append({
        "ANO4": anio, "TRIMESTRE": trimestre, "n_hogares": len(resultado),
        "pct_region_nula": 100 * resultado["REGION"].isna().mean(),
        "pct_unidades_adulto_equivalente_na": 100 * resultado["unidades_adulto_equivalente"].isna().mean(),
        "pct_ingreso_no_declarado": 100 * ingreso_no_declarado_true.mean(),
    })
    particion = DISTANCIA_LINEA_POBREZA / f"ANO4={anio}" / f"TRIMESTRE={trimestre}" / "data.parquet"
    particion.parent.mkdir(parents=True, exist_ok=True)
    resultado[COLUMNAS_DISCO].reset_index(drop=True).to_parquet(particion, index=False)
    n_filas_totales += len(resultado)

duracion_loop = time.time() - t0
cobertura_historica = pd.DataFrame(filas_cobertura)
print(f"{len(TRIMESTRES_HISTORICOS)} trimestres procesados, {n_filas_totales:,} filas escritas, "
      f"{duracion_loop:.1f}s")

### Trimestres faltantes, y motivo

Se confirma contra el rango completo cuáles huecos de publicación caen dentro de la
ventana histórica, sin asumirlos.

In [ ]:
rango_completo_2003t3_2015t2 = [(a, t) for a in range(2003, 2016) for t in range(1, 5)
                                if (2003, 3) <= (a, t) <= (2015, 2)]
faltantes_historicos = [p for p in rango_completo_2003t3_2015t2 if p not in set(TRIMESTRES_HISTORICOS)]
print(f"faltantes dentro de 2003T3-2015T2: {faltantes_historicos}")

### Registro de la corrida y tabla de cobertura por trimestre

In [ ]:
pm.registrar_corrida(
    META / "corridas.csv", "05c_distancia_historica",
    {"ERA": ERA, "TRIMESTRES": TRIMESTRES_HISTORICOS},
    len(TRIMESTRES_HISTORICOS), n_filas_totales, duracion_loop, COMMIT_GIT,
)
pd.set_option("display.max_rows", None)
cobertura_historica

## Validaciones

### Quiebre de aglomerados (verificado en los datos, no asumido)

Se verifica cuántos aglomerados hay antes y después de la ampliación de cobertura de
la EPH, contra lo que documenta la literatura de referencia.

In [ ]:
def aglomerados_en(anio: int, trimestre: int) -> set[int]:
    p = ARMONIZADO / "hogar" / f"ANO4={anio}" / f"TRIMESTRE={trimestre}" / "data.parquet"
    df = pd.read_parquet(p, columns=["AGLOMERADO"])
    return set(pd.to_numeric(df["AGLOMERADO"], errors="coerce").dropna().astype(int).unique())


aglomerados_2006t2 = aglomerados_en(2006, 2)
aglomerados_2006t3 = aglomerados_en(2006, 3)
nuevos_2006t3 = sorted(aglomerados_2006t3 - aglomerados_2006t2)

estables_2003t3_2006t2 = set.intersection(*(aglomerados_en(a, t) for a, t in (
    (2003, 3), (2003, 4), (2004, 1), (2004, 2), (2004, 3), (2004, 4),
    (2005, 1), (2005, 2), (2005, 3), (2005, 4), (2006, 1), (2006, 2))))

print(f"aglomerados en 2006T2: {len(aglomerados_2006t2)}")
print(f"aglomerados en 2006T3: {len(aglomerados_2006t3)}")
print(f"códigos nuevos en 2006T3: {nuevos_2006t3}")
print(f"aglomerados estables en todos los trimestres 2003T3-2006T2: {len(estables_2003t3_2006t2)}")

**Confirmado**: el quiebre de cobertura cruda de la EPH coincide con los códigos de
aglomerado que Tornarolli excluye de su serie comparable -- aunque el panel restringido
que usa la literatura de referencia no coincide exactamente en cantidad con el nuestro.
Esa diferencia queda marcada como límite conocido, no resuelto; la validación siguiente
usa el criterio verificable (excluir los códigos nuevos de toda la serie).

### Otros cortes conocidos que atraviesa esta ventana

Los huecos de publicación y el período de intervención INDEC ya documentados
atraviesan esta ventana -- ninguno se oculta ni se corrige acá.

### Tasas semestrales vs. CEDLAS

Lee directamente las particiones ya escritas en `data/02b_distancia_linea_pobreza/hogar/`, no
recalcula -- valida la capa que quedó en disco, no una versión en memoria distinta.

In [ ]:
AGLOMERADOS_EXCLUIDOS_COMPARABLE = {38, 91, 93}


tasa_ponderada = distancia_linea_pobreza.tasa_ponderada


def cargar_particion_historica_comparable(anio: int, trimestre: int) -> pd.DataFrame:
    p = DISTANCIA_LINEA_POBREZA / f"ANO4={anio}" / f"TRIMESTRE={trimestre}" / "data.parquet"
    df = pd.read_parquet(p)
    df = df[~df["AGLOMERADO"].isin(AGLOMERADOS_EXCLUIDOS_COMPARABLE)].copy()
    df["peso_personas"] = df["peso_hogar"] * df["n_miembros"]
    return df


def tasa_semestre_historico(componentes: tuple[tuple[int, int], ...]) -> dict:
    df = pd.concat([cargar_particion_historica_comparable(a, t) for a, t in componentes], ignore_index=True)
    return {"pobreza_personas": tasa_ponderada(df, "pobre", "peso_personas"),
            "indigencia_personas": tasa_ponderada(df, "indigente", "peso_personas")}


semestres_historicos_computables = cr.semestres_historicos_computables


SEMESTRES_HIST = semestres_historicos_computables(TRIMESTRES_HISTORICOS)
filas_propias = [{"ANO4": a, "SEMESTRE": s, "semestre_movil": m, **tasa_semestre_historico(c)}
                  for a, s, c, m in SEMESTRES_HIST]
tasas_propias = pd.DataFrame(filas_propias)

cedlas = pd.read_csv(RETROPOLACION / "cedlas" / "series_comparables_anexo_b.csv")
cedlas = cedlas.rename(columns={c: c.replace("indig_", "indigencia_").replace("pob_", "pobreza_")
                                 for c in cedlas.columns}).drop(columns=["semestre", "semestre_movil"])

comparacion_cedlas = tasas_propias.merge(cedlas, on=["ANO4", "SEMESTRE"], how="inner")
comparacion_cedlas["brecha_pobreza_pp"] = (
    comparacion_cedlas["pobreza_personas"] - comparacion_cedlas["pobreza_serie_comparable"])
comparacion_cedlas["brecha_indigencia_pp"] = (
    comparacion_cedlas["indigencia_personas"] - comparacion_cedlas["indigencia_serie_comparable"])

print(f"{len(comparacion_cedlas)} semestres comparados (de {len(tasas_propias)} propios)")
print(f"brecha media pobreza: {comparacion_cedlas['brecha_pobreza_pp'].mean():+.2f}pp "
      f"(desvío {comparacion_cedlas['brecha_pobreza_pp'].std():.2f}pp)")
print(f"brecha media indigencia: {comparacion_cedlas['brecha_indigencia_pp'].mean():+.2f}pp "
      f"(desvío {comparacion_cedlas['brecha_indigencia_pp'].std():.2f}pp)")
comparacion_cedlas[["ANO4", "SEMESTRE", "semestre_movil", "pobreza_personas", "pobreza_serie_comparable",
                    "brecha_pobreza_pp", "indigencia_personas", "indigencia_serie_comparable",
                    "brecha_indigencia_pp"]]

### Frontera histórico/regular: tasas a ambos lados vs. CEDLAS

No se compara el nivel de la canasta entre ambas eras (ese salto es inflación acumulada
del hueco sin publicar entre ellas, y la continuidad de la canasta en sí ya está
validada en `05b`) -- se compara la tasa de pobreza de cada lado contra CEDLAS para el
mismo semestre, que cubre ambos con un método único.

In [ ]:
def cargar_particion_oficial_comparable(anio: int, trimestre: int) -> pd.DataFrame:
    p = DISTANCIA_LINEA_POBREZA / f"ANO4={anio}" / f"TRIMESTRE={trimestre}" / "data.parquet"
    df = pd.read_parquet(p)
    individual = pd.read_parquet(NUCLEO / "individual" / f"ANO4={anio}" / f"TRIMESTRE={trimestre}" / "data.parquet",
                                  columns=["CODUSU", "NRO_HOGAR"])
    n_miembros = individual.groupby(["CODUSU", "NRO_HOGAR"]).size().rename("n_miembros")
    idx = pd.MultiIndex.from_frame(df[["CODUSU", "NRO_HOGAR"]])
    df["n_miembros"] = n_miembros.reindex(idx).to_numpy()
    df = df[~df["AGLOMERADO"].isin(AGLOMERADOS_EXCLUIDOS_COMPARABLE)]
    df = df[~(df["ingreso_no_declarado"] == True)].copy()  # noqa: E712
    df["peso_personas"] = df["PONDIH"] * df["n_miembros"]
    return df


def tasa_semestre_oficial(componentes: tuple[tuple[int, int], ...]) -> dict:
    df = pd.concat([cargar_particion_oficial_comparable(a, t) for a, t in componentes], ignore_index=True)
    return {"pobreza_personas": tasa_ponderada(df, "pobre", "peso_personas"),
            "indigencia_personas": tasa_ponderada(df, "indigente", "peso_personas")}


tasa_2015_1 = tasa_semestre_historico(((2015, 1), (2015, 2)))
tasa_2016_2 = tasa_semestre_oficial(((2016, 3), (2016, 4)))

fila_2015_1 = cedlas[(cedlas["ANO4"] == 2015) & (cedlas["SEMESTRE"] == 1)].iloc[0]
fila_2016_2 = cedlas[(cedlas["ANO4"] == 2016) & (cedlas["SEMESTRE"] == 2)].iloc[0]

frontera = pd.DataFrame([
    {"semestre": "2015-I (histórica)", "pobreza_personas": tasa_2015_1["pobreza_personas"],
     "pobreza_cedlas": fila_2015_1["pobreza_serie_comparable"],
     "indigencia_personas": tasa_2015_1["indigencia_personas"],
     "indigencia_cedlas": fila_2015_1["indigencia_serie_comparable"]},
    {"semestre": "2016-II (regular/oficial)", "pobreza_personas": tasa_2016_2["pobreza_personas"],
     "pobreza_cedlas": fila_2016_2["pobreza_serie_comparable"],
     "indigencia_personas": tasa_2016_2["indigencia_personas"],
     "indigencia_cedlas": fila_2016_2["indigencia_serie_comparable"]},
])
frontera["brecha_pobreza_pp"] = frontera["pobreza_personas"] - frontera["pobreza_cedlas"]
frontera["brecha_indigencia_pp"] = frontera["indigencia_personas"] - frontera["indigencia_cedlas"]
frontera.to_csv(META / "frontera_2015_2016_vs_cedlas.csv", index=False)
frontera

## Línea de pobreza/indigencia consolidada por trimestre-región (ambas eras)

Extracto de referencia, sin volver a calcular nada -- se deduplica y se escribe en
`data/meta/linea_pobreza_trimestral.csv` (versionado) para tener la línea a mano sin
leer el Parquet por hogar.

In [ ]:
partes_linea = []
for p in sorted(DISTANCIA_LINEA_POBREZA.glob("ANO4=*/TRIMESTRE=*/data.parquet")):
    partes_linea.append(pd.read_parquet(p, columns=["ANO4", "TRIMESTRE", "REGION", "era", "cba_adulto_equivalente", "cbt_adulto_equivalente"]))
linea_larga = pd.concat(partes_linea, ignore_index=True)

n_valores_distintos = linea_larga.groupby(["ANO4", "TRIMESTRE", "REGION", "era"])[["cba_adulto_equivalente", "cbt_adulto_equivalente"]].nunique().max()
assert (n_valores_distintos == 1).all(), "cba_adulto_equivalente/cbt_adulto_equivalente no es constante por trimestre-región-era -- revisar antes de deduplicar"

linea_pobreza_trimestral = (
    linea_larga.drop_duplicates(subset=["ANO4", "TRIMESTRE", "REGION", "era"])
    .rename(columns={"cba_adulto_equivalente": "linea_indigencia_adulto_equivalente", "cbt_adulto_equivalente": "linea_pobreza_adulto_equivalente"})
    .sort_values(["ANO4", "TRIMESTRE", "REGION"])
    .reset_index(drop=True)
)
linea_pobreza_trimestral.to_csv(META / "linea_pobreza_trimestral.csv", index=False)

print(f"{len(linea_pobreza_trimestral)} filas trimestre×región×era")
linea_pobreza_trimestral["era"].value_counts()